In [3]:
# ==============================================================================
# CELDA 1 (COMPLETA): INGESTA ASISTENTES + DOCENTES 2018-2019
# ==============================================================================
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PERSONAL/'

archivos_asist = {2018: 'Resumen_Asistentes_2018.csv', 2019: 'Resumen_Asistentes_2019.csv'}
archivos_doc = {2018: 'Dotacion_docente_2018.csv', 2019: 'Dotacion_docente_2019.csv'}

def cargar(archivos):
    brutas = {}
    for anio, nombre in archivos.items():
        ruta = RUTA + nombre
        for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
            try:
                brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
                break
            except UnicodeDecodeError:
                continue
        print(f"OK | {anio} | {brutas[anio].shape[0]} filas x {brutas[anio].shape[1]} columnas")
    return brutas

print("--- Asistentes ---")
asist_brutas = cargar(archivos_asist)
print("\n--- Docentes ---")
doc_brutas = cargar(archivos_doc)

print("\nColumnas Docentes 2018:", doc_brutas[2018].columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Asistentes ---
OK | 2018 | 16041 filas x 26 columnas
OK | 2019 | 16182 filas x 27 columnas

--- Docentes ---
OK | 2018 | 16041 filas x 46 columnas
OK | 2019 | 16181 filas x 47 columnas

Columnas Docentes 2018: ['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'COD_REG_RBD', 'COD_PRO_RBD', 'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD', 'NOM_DEPROV_RBD', 'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'ESTADO_ESTAB', 'DC_A', 'HH_A', 'DC_UTP', 'HH_UTP', 'DC_PDIR', 'HH_PDIR', 'DC_DIR', 'HH_DIR', 'DC_OES', 'HH_OES', 'DC_OF', 'HH_OF', 'DC_JUTP', 'HH_JUTP', 'DC_IG', 'HH_IG', 'DC_OR', 'HH_OR', 'DC_DIR_SOST', 'HH_DIR_SOST', 'DC_TP_SOST', 'HH_TP_SOST', 'DC_SUP_SOST', 'HH_SUP_SOST', 'DC_SUBDIR', 'HH_SUBDIR', 'DC_PROF_ENC', 'HH_PROF_ENC', 'DC_EDUC_TRAD', 'HH_EDUC_TRAD', 'DC_TOT', 'HH_TOT']


In [4]:
# ==============================================================================
# CELDA 2: PROCESAR ASISTENTES + DOCENTES, UNIR EN UN SOLO NOTEBOOK
# ==============================================================================
def procesar_asistentes(df):
    d = df.copy()
    d['RBD'] = pd.to_numeric(d['RBD'], errors='coerce')
    d = d.dropna(subset=['RBD'])
    d['rbd'] = d['RBD'].astype('Int64').astype(str)
    d['n_asistentes'] = pd.to_numeric(d['N_ASIS'], errors='coerce')
    return d[['rbd', 'n_asistentes']]

def procesar_docentes(df):
    d = df.copy()
    d['RBD'] = pd.to_numeric(d['RBD'], errors='coerce')
    d = d.dropna(subset=['RBD'])
    d['rbd'] = d['RBD'].astype('Int64').astype(str)
    d['n_docentes'] = pd.to_numeric(d['DC_TOT'], errors='coerce')
    d['horas_docentes'] = pd.to_numeric(d['HH_TOT'], errors='coerce')
    d['n_directivos'] = pd.to_numeric(d['DC_DIR'], errors='coerce').fillna(0) + pd.to_numeric(d['DC_PDIR'], errors='coerce').fillna(0)
    return d[['rbd', 'n_docentes', 'horas_docentes', 'n_directivos']]

asist_limpios = {a: procesar_asistentes(df).groupby('rbd', as_index=False).mean() for a, df in asist_brutas.items()}
doc_limpios = {a: procesar_docentes(df).groupby('rbd', as_index=False).mean() for a, df in doc_brutas.items()}

asist_1819 = pd.concat(asist_limpios.values(), ignore_index=True).groupby('rbd', as_index=False).mean()
doc_1819 = pd.concat(doc_limpios.values(), ignore_index=True).groupby('rbd', as_index=False).mean()

# Unimos ambas fuentes en un solo DataFrame de personal
personal_1819 = pd.merge(doc_1819, asist_1819, on='rbd', how='outer')

print(f"Docentes: {doc_1819.shape[0]} colegios | Asistentes: {asist_1819.shape[0]} colegios")
print(f"Personal combinado: {personal_1819.shape[0]} colegios")
print(personal_1819.describe())

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
personal_1819.to_parquet(RUTA_SALIDA + 'personal_2018_19_por_rbd.parquet', index=False)
print("Guardado OK")

Docentes: 16182 colegios | Asistentes: 16183 colegios
Personal combinado: 16183 colegios
         n_docentes  horas_docentes  n_directivos  n_asistentes
count  16182.000000    16182.000000  16182.000000  16183.000000
mean      16.296317      586.079780      0.790848     11.410462
std       21.598468      803.980583      0.918463     15.037726
min        0.000000        0.000000      0.000000      0.000000
25%        0.000000        0.000000      0.000000      0.000000
50%        7.000000      230.250000      1.000000      5.500000
75%       25.000000      889.500000      1.000000     17.500000
max      225.000000     9891.000000     12.000000    173.000000
Guardado OK


In [5]:
# ==============================================================================
# CELDA 3: INTEGRAR PERSONAL A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v9.parquet')
personal = pd.read_parquet(RUTA + 'personal_2018_19_por_rbd.parquet')

df_modelo_v10 = pd.merge(df_modelo, personal, on='rbd', how='left', validate='one_to_one')

print(f"Filas: {len(df_modelo_v10)} (antes: {len(df_modelo)})")
print(f"Con dato de personal: {df_modelo_v10['n_docentes'].notna().sum()}")

df_modelo_v10.to_parquet(RUTA + 'tabla_modelo_final_v10.parquet', index=False)
print(df_modelo_v10.shape)

Filas: 7754 (antes: 7754)
Con dato de personal: 7754
(7754, 64)
